In [0]:

# Databricks notebook source

from pyspark.sql.functions import *

CATALOG = "adbsmartdatatdp"
SILVER = "silver"
GOLD = "gold"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# SALES SUMMARY
# ==========================================================

sales = spark.table(f"{CATALOG}.{SILVER}.sales")
customers = spark.table(f"{CATALOG}.{SILVER}.customers")
products = spark.table(f"{CATALOG}.{SILVER}.products")

sales_summary = (
    sales.groupBy("sale_date")
    .agg(
        round(sum("total"), 2).alias("total_sales"),
        count("*").alias("total_transactions"),
        round(avg("total"), 2).alias("average_ticket")
    )
)

sales_summary.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(
    f"{CATALOG}.{GOLD}.sales_summary"
)

# ==========================================================
# SALES BY CITY
# ==========================================================

sales_by_city = (
    sales.join(customers, "customer_id")
    .groupBy("city")
    .agg(
        round(sum("total"),2).alias("total_sales"),
        count("*").alias("total_transactions")
    )
)

sales_by_city.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(
    f"{CATALOG}.{GOLD}.sales_by_city"
)

# ==========================================================
# SALES BY CATEGORY
# ==========================================================

sales_by_category = (
    sales.join(products, "product_id")
    .groupBy("category")
    .agg(
        round(sum("total"),2).alias("total_sales"),
        count("*").alias("total_transactions")
    )
)

sales_by_category.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(
    f"{CATALOG}.{GOLD}.sales_by_category"
)

# ==========================================================
# TOP CUSTOMERS
# ==========================================================

top_customers = (
    sales.join(customers, "customer_id")
    .groupBy("customer_id","full_name")
    .agg(
        round(sum("total"),2).alias("total_amount"),
        count("*").alias("total_orders")
    )
    .orderBy(desc("total_amount"))
)

top_customers.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(
    f"{CATALOG}.{GOLD}.top_customers"
)

# ==========================================================
# TOP PRODUCTS
# ==========================================================

top_products = (
    sales.join(products, "product_id")
    .groupBy("product_id","product_name")
    .agg(
        sum("quantity").alias("units_sold"),
        round(sum("total"),2).alias("total_sales")
    )
    .orderBy(desc("units_sold"))
)

top_products.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(
    f"{CATALOG}.{GOLD}.top_products"
)

# ==========================================================
# KPI DASHBOARD
# ==========================================================

kpi_dashboard = (
    sales.groupBy("sale_date")
    .agg(
        round(sum("total"),2).alias("total_revenue"),
        countDistinct("customer_id").alias("total_customers"),
        countDistinct("product_id").alias("total_products"),
        round(avg("total"),2).alias("average_ticket")
    )
    .withColumnRenamed("sale_date","report_date")
)

kpi_dashboard.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(
    f"{CATALOG}.{GOLD}.kpi_dashboard"
)

print("GOLD BUILD COMPLETED SUCCESSFULLY")
